### NPU 실행모델 변환과 에뮬레이션
- 파일 : n2-2_dfc.ipynb
- 역할 : DFC로 ONNX를 파싱·양자화·에뮬레이션·컴파일해 NPU 실행모델(HEF)을 만드는 전 과정을 수행합니다.
- 기능 :
    1. DFC 전용 환경 준비  
    2. 변환할 ONNX 모델 준비  
    3. 파싱 스크립트 작성과 실행  
    4. Calibration·검증 데이터 준비  
    5. 양자화  
    6. 에뮬레이션 실행  
    7. Top-1 정확도와 예측 일치율 계산  
    8. 양자화 수준 2와 에뮬레이션  
    9. HEF 컴파일  
    10. HEF 확인과 보관  


>> [코드 5-1] DFC 전용 환경 준비 (n2-2_dfc.ipynb, STEP 1)
<hr>

In [ ]:
#- 모듈로딩과 Drive 연결
import os, subprocess
from google.colab import drive

#- Drive 연결
drive.mount('/content/drive')

#- DFC 파일 경로 설정
HAILO_FILE = "hailo_dataflow_compiler-3.34.0-py3-none-linux_x86_64.whl"
WHL = f"/content/drive/MyDrive/hailo/{HAILO_FILE}"

#- 기본 환경 패키지와 uv 설치
!pip install -q uv onnx onnxruntime "protobuf<6"

#- DFC 전용 3.10 환경 생성과 설치
!uv venv --clear --python 3.10 /content/dfc310
!uv pip install --python /content/dfc310/bin/python "$WHL" 2>&1 | tail -n 1

#- DFC 스크립트를 전용 환경으로 실행하는 도우미
PY310 = "/content/dfc310/bin/python"

#- DFC 코드는 3.10 전용 환경에서 실행
def run310(script):
    p = subprocess.run([PY310, script], capture_output=True, text=True)
    print(p.stdout[-3000:])
    if p.returncode != 0:
        print("----- 오류 -----")
        print(p.stderr[-3000:])

# 확인 포인트 -------------------------------------------------------------------------
# - 마지막 줄에 설치된 패키지 목록 출력 체크 (약 75초)

>> [코드 5-2] 변환할 ONNX 모델 준비 (STEP 2)
<hr>

In [ ]:
#- 모듈로딩
import torch
from torchvision import models

#- 파일 상수: 변환 산출물 (파싱 단계의 입력)
ONNX_FILE = "mobilenet_v2.onnx"

#- 원본 모델 로딩과 ONNX 변환 (Opset 13)
m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1).eval()
torch.onnx.export(m, torch.randn(1, 3, 224, 224), ONNX_FILE,
                  opset_version=13,
                  input_names=["input"], output_names=["output"],
                  dynamo=False)

print(f"ONNX 크기: {os.path.getsize(ONNX_FILE)/1e6:.1f} MB")

# 확인 포인트 -------------------------------------------------------------------------
# - ONNX 크기 출력 체크

**실행 결과 예시**
```
ONNX 크기: 14.0 MB
```

>> [코드 5-3] 파싱 스크립트 작성과 실행 (STEP 3)
<hr>

In [ ]:
%%writefile s1_parse.py

#- 모듈 로딩
from hailo_sdk_client import ClientRunner

#- 파일 및 칩 상수: 입력 ONNX(변환 산출물)·출력 HAR(파싱 결과), 모델 이름
CHIP = "hailo8"
ONNX_FILE = "mobilenet_v2.onnx"
PARSED_HAR = "mobilenet_v2_parsed.har"
NET_NAME = "mobilenet_v2"

#- 대상 칩 지정
runner = ClientRunner(hw_arch= CHIP)

# ONNX → 내부 계층 번역 (입출력 노드와 shape 지정)
runner.translate_onnx_model(  ONNX_FILE, NET_NAME,
                              start_node_names=["input"], end_node_names=["output"],
                              net_input_shapes={"input": [1, 3, 224, 224]} )
#- 파싱 결과 저장
runner.save_har(PARSED_HAR)
print("PARSE_OK")

# 확인 포인트 ------------------------------------------------------------------------
# - Writing s1_parse.py 출력 체크

>> [코드 5-3 실행] 파싱 스크립트 실행

In [ ]:
#-  방금 저장한 스크립트를 전용 환경(3.10)으로 실행
run310("s1_parse.py")

# 확인 포인트 ------------------------------------------------------------------------
# - PARSE_OK 출력 체크 (mobilenet_v2_parsed.har 생성)

**실행 결과 예시**
```
[info] Translation started on ONNX model mobilenet_v2
[info] Start nodes mapped from original model: 'input': 'mobilenet_v2/input_layer1'.
[info] End nodes mapped from original model: '/classifier/classifier.1/Gemm'.
[info] Translation completed on ONNX model mobilenet_v2 (completion time: 00:00:01.05)
[info] Saved HAR to: /content/mobilenet_v2_parsed.har
PARSE_OK
```

>> [코드 5-4] Calibration·검증 데이터 준비 (STEP 4)
<hr>

In [ ]:
#- 모듈로딩
import urllib.request, tarfile, glob, numpy as np, json
from PIL import Image

#- Imagenette 다운로드 상수와 경로
DS_ORG, DS_SAVE = "imagenette2-160.tgz", "imagenette.tgz"
URL = f"https://s3.amazonaws.com/fast-ai-imageclas/{DS_ORG}"
DS_DIR = "imagenette2-160"
TRAIN_DIR = f"{DS_DIR}/train"
VALID_DIR = f"{DS_DIR}/val"

#- 다운로드와 압축 해제 (한 번만)
if not os.path.exists(DS_DIR):
    urllib.request.urlretrieve(URL, DS_SAVE)
    with tarfile.open(DS_SAVE) as t:
        t.extractall(".", filter="data")

#- NHWC·0~255 로딩 (정규화는 모델 스크립트가 담당)
def load(files):
    return np.stack([np.asarray(Image.open(f).convert("RGB").resize((224, 224)),
                                dtype=np.float32) for f in files])

#- 학습용과 검증용 이미지 추출
train = sorted(glob.glob( f"{TRAIN_DIR}/*/*.JPEG"))
val = sorted(glob.glob(f"{VALID_DIR}/*/*.JPEG"))[::40]

#- Calibration 64장과 검증 세트 저장
np.save("calib_64.npy", load(train[::len(train)//64][:64]))
np.save("val.npy", load(val))
json.dump([f.split("/")[-2] for f in val], open("val_wnids.json", "w"))

print("calib", np.load("calib_64.npy").shape, "/ val", np.load("val.npy").shape)

# 확인 포인트 -------------------------------------------------------------------------
# - calib와 val 배열의 shape 출력 체크

**실행 결과 예시**
```
calib (64, 224, 224, 3) / val (99, 224, 224, 3)
```

>> [코드 5-5] 양자화(최적화) 실행: 수준 0 (STEP 5)
<hr>

In [ ]:
%%writefile s2_optimize.py

#- 모듈로딩
import numpy as np
from hailo_sdk_client import ClientRunner

#- 파일 상수: 파싱 HAR·Calibration 데이터(입력), 양자화 HAR(출력)
PARSED_HAR = "mobilenet_v2_parsed.har"
CALIB_FILE = "calib_64.npy"
HAR_FILE = "mobilenet_v2_quantized.har"

#- 파싱 결과 로딩
runner = ClientRunner(har=PARSED_HAR)

#- 모델 스크립트: 입력 정규화 내장 + 최적화 수준 0
runner.load_model_script(
    "normalization1 = normalization([123.675, 116.28, 103.53], [58.395, 57.12, 57.375])\n"
    "model_optimization_flavor(optimization_level=0, compression_level=0)\n"
)

#- Calibration 데이터로 양자화
runner.optimize(np.load(CALIB_FILE))

#- 양자화 결과 저장
runner.save_har(HAR_FILE)
print("OPTIMIZE_OK")

# 확인 포인트 -------------------------------------------------------------------------
# - Writing s2_optimize.py 출력 체크

>> [코드 5-5 실행] 양자화 스크립트 실행

In [ ]:
#-  방금 저장한 스크립트를 전용 환경(3.10)으로 실행
run310("s2_optimize.py")

# 확인 포인트 ------------------------------------------------------------------------
# - OPTIMIZE_OK 출력 체크 (mobilenet_v2_quantized.har 생성, 무료 Colab 약 90초)

**실행 결과 예시**
```
[info] Starting Model Optimization
[warning] Running model optimization with zero level of optimization is not recommended ...
[info] Using dataset with 64 entries for calibration
[info] Model Optimization Algorithm Statistics Collector is done (completion time is 00:00:27.65)
[info] Bias Correction skipped
[info] Adaround skipped
[info] Quantization-Aware Fine-Tuning skipped
[info] Model Optimization is done
OPTIMIZE_OK
```

>> [코드 5-6] 에뮬레이션 실행 (STEP 6)
<hr>

In [ ]:
%%writefile s3_emulate.py

#- 모듈 로딩
import json, numpy as np
from hailo_sdk_client import ClientRunner, InferenceContext

#- 파일 상수: 검증 데이터 · 양자화 HAR(입력 2개), 예측 결과(출력 1개)
VAL_FILE, HAR_FILE, PRED_FILE = "val.npy", "mobilenet_v2_quantized.har", "pred.json"

#- 검증 데이터와 양자화 결과 로딩
val = np.load(VAL_FILE)
runner = ClientRunner(har=HAR_FILE)

#- 비교할 두 추론 모드
MODES = [("fp", InferenceContext.SDK_FP_OPTIMIZED),
         ("q0", InferenceContext.SDK_QUANTIZED)]

#- 모드별 예측을 담을 딕셔너리
pred = {}

#- FP 최적화 모드와 양자화 모드로 에뮬레이션
for name, ctx in MODES :
    # 해당 모드의 실행 문맥에서 일괄 추론
    with runner.infer_context(ctx) as c:
        out = runner.infer(c, val)

    # 로짓에서 Top-1 인덱스 추출
    pred[name] = np.array(out).reshape(len(val), -1).argmax(1).tolist()

#- 예측 결과 저장
json.dump(pred, open(PRED_FILE, "w"))
print("EMULATE_OK")

# 확인 포인트 -------------------------------------------------------------------------
# - Writing s3_emulate.py 출력 체크

>> [코드 5-6 실행] 에뮬레이션 스크립트 실행

In [ ]:
#-  방금 저장한 스크립트를 전용 환경(3.10)으로 실행
run310("s3_emulate.py")

# 확인 포인트 ------------------------------------------------------------------------
# - EMULATE_OK 출력 체크 (pred.json 저장)

>> [코드 5-7] Top-1 정확도와 예측 일치율 계산 (STEP 7)
<hr>

In [ ]:
#- Imagenette 폴더명과 ImageNet 인덱스 매핑
IDX = {"n01440764": 0,   "n02102040": 217, "n02979186": 482, "n03000684": 491,
       "n03028079": 497, "n03394916": 566, "n03417042": 569, "n03425413": 571,
       "n03445777": 574, "n03888257": 701}

#- 정답과 예측 로딩
gt = np.array([IDX[w] for w in json.load(open("val_wnids.json"))])
pred = {k: np.array(v) for k, v in json.load(open("pred.json")).items()}

#- 정확도와 두 모드의 예측 일치율
for k in pred: print(k, f"Top-1 {np.mean(pred[k] == gt)*100:.1f}%")
print(f"예측 일치율 {np.mean(pred['fp'] == pred['q0'])*100:.1f}%")

# 확인 포인트 -------------------------------------------------------------------------
# - fp와 q0의 Top-1, 예측 일치율(%) 출력 체크

**실행 결과 예시**
```
fp Top-1 79.8%
q0 Top-1 70.7%
예측 일치율 80.8%
```

>> [코드 5-8] 양자화 수준 2와 에뮬레이션 (STEP 8)
<hr>

In [ ]:
%%writefile s2b_optimize_l2.py

#- 모듈 로딩
import json, numpy as np
from hailo_sdk_client import ClientRunner, InferenceContext

#- 파일 상수: 입력(파싱 HAR·Calibration·검증 데이터·기존 예측), 출력(수준 2 HAR)
PARSED_HAR = "mobilenet_v2_parsed.har"
CALIB_FILE = "calib_64.npy"
VAL_FILE = "val.npy"
L2_HAR = "mobilenet_v2_quantized_l2.har"
PRED_FILE = "pred.json"

#- 파싱 결과에서 다시 시작: 최적화 수준 2
runner = ClientRunner(har=PARSED_HAR)
runner.load_model_script(
    "normalization1 = normalization([123.675, 116.28, 103.53], [58.395, 57.12, 57.375])\n"
    "model_optimization_flavor(optimization_level=2, compression_level=0)\n"
)
# 수준 2로 다시 양자화 (오차 보정 포함, 시간 소요)
runner.optimize(np.load(CALIB_FILE))
runner.save_har(L2_HAR)

#- 수준 2 결과로 에뮬레이션
with runner.infer_context(InferenceContext.SDK_QUANTIZED) as c:
    out = runner.infer(c, np.load(VAL_FILE))

pred = json.load(open(PRED_FILE))

# q2 예측을 기존 결과에 추가
pred["q2"] = np.array(out).reshape(len(out), -1).argmax(1).tolist()
json.dump(pred, open(PRED_FILE, "w"))
print("OPTIMIZE_L2_OK")

# 확인 포인트 -------------------------------------------------------------------------
# - 무료 Colab에서 약 9분 소요
# - 진행 로그가 계속 출력되면 정상

>> [코드 5-8 실행] 수준 2 양자화 스크립트 실행

In [ ]:
#-  방금 저장한 스크립트를 전용 환경(3.10)으로 실행
run310("s2b_optimize_l2.py")
# 확인 포인트 ------------------------------------------------------------------------
# - OPTIMIZE_L2_OK 출력 체크 (무료 Colab 약 9분, 진행 로그 출력은 정상)

**실행 결과 예시**
```
Epoch 1/4 … _distill_loss_mobilenet_v2/fc1: 0.2149
Epoch 4/4 … _distill_loss_mobilenet_v2/fc1: 0.1972
[info] Model Optimization Algorithm Quantization-Aware Fine-Tuning is done (completion time is 00:04:18.39)
[info] Output layers signal-to-noise ratio (SNR): measures the quantization noise (higher is better)
[info]   mobilenet_v2/output_layer1 SNR:  14.54 dB
OPTIMIZE_L2_OK
```

>> [코드 5-9] HEF 컴파일 (STEP 9)
<hr>

In [ ]:
%%writefile s4_compile.py

#- 모듈 로딩
from hailo_sdk_client import ClientRunner

#- 파일 상수: 입력 HAR(수준 2)·출력 HEF(NPU 실행모델)
HAR_FILE = "mobilenet_v2_quantized_l2.har"
HEF_FILE = "mobilenet_v2.hef"

#- 수준 2 양자화 HAR를 NPU 실행모델(HEF)로 컴파일
runner = ClientRunner(har=HAR_FILE)

#- 대상 NPU 자원에 계층을 배치하고 HEF로 컴파일 (수 분 소요)
hef = runner.compile()

#- 생성된 HEF 바이너리 저장
open(HEF_FILE, "wb").write(hef)
print("COMPILE_OK")

# 확인 포인트 -------------------------------------------------------------------------
# - Writing s4_compile.py 출력 체크

>> [코드 5-9 실행] 컴파일 스크립트 실행

In [ ]:
#-  방금 저장한 스크립트를 전용 환경(3.10)으로 실행
run310("s4_compile.py")

# 확인 포인트 ------------------------------------------------------------------------
# - COMPILE_OK 출력 체크
# - mobilenet_v2.hef 생성 확인
# - 무료 Colab 약 3분 정도 소요 될 수 있음

**실행 결과 예시**
```
[info] Successful Mapping (allocation time: 2m 21s)
[info] | Total     | 75%                 | 39.8%               | 28.7%              |
[info] Bandwidth of model inputs: 1.14844 Mbps, outputs: 0.00762939 Mbps (for a single frame)
[info] Bandwidth of DDR buffers: 0.0 Mbps (for a single frame)
[info] Successful Compilation (compilation time: 8s)
COMPILE_OK
```

>> [코드 5-10] HEF 확인과 보관 (STEP 10)
<hr>

In [ ]:
#- 파일 상수: NPU 실행모델
HEF_FILE = "mobilenet_v2.hef"

#- HEF 크기 확인과 Drive 보관
print(f"HEF 크기: {os.path.getsize(HEF_FILE)/1e6:.2f} MB")
!cp $HEF_FILE /content/drive/MyDrive/hailo/

# 확인 포인트 -------------------------------------------------------------------------
# - HEF 크기 출력 확인
# - Drive의 hailo 폴더에 HEF 파일이 저장되었는지 확인 (세션이 끝나도 보존)

>> [보관] 수준 2 HAR을 Drive에 저장 (노트북 전용 셀)

In [ ]:
#- 수준 2 HAR도 Drive에 보관: 미니프로젝트(NPU 에뮬레이터 백엔드)가 사용
!cp mobilenet_v2_quantized_l2.har /content/drive/MyDrive/hailo/
!ls -lh /content/drive/MyDrive/hailo/

# 확인 포인트 ------------------------------------------
# - hailo 폴더에 hef와 l2 har 두 파일 표시 체크

**실행 결과 예시**
```
HEF 크기: 5.06 MB
```